In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import roc_curve, auc, confusion_matrix
import os

# ---------------------------------------------------------
# DIRECTORY SETUP (Relative paths for reviewer reproducibility)
# ---------------------------------------------------------
# Assuming this script is executed from within the /Source_Code folder
data_dir = "../Data/Publication_Results"
output_dir = "../Data/Publication_Results"

class_names = ['Normal', 'Scar', 'Inflamed', 'Tumor']
models_list = ['Bio', 'Mech', 'Fused']
model_titles = ['Biochemical Baseline', 'Biomechanical Baseline', 'Multimodal Fusion']
colors = ['#1f77b4', '#2ca02c', '#d62728']

# ---------------------------------------------------------
# NATURE BME TYPOGRAPHICAL FORMATTING
# ---------------------------------------------------------
plt.rcParams.update({
    'font.size': 10,
    'font.family': 'sans-serif',
    'axes.linewidth': 1.2,
    'xtick.major.width': 1.2,
    'ytick.major.width': 1.2,
    'pdf.fonttype': 42 # CRITICAL: Ensures editable vector text in Adobe Illustrator
})

print(f"Reading raw publication data from: {data_dir}")
print("Generating 10 Master Figures and Exporting Raw Matrices...\n")

df_roc = pd.read_csv(os.path.join(data_dir, "Ablation_ROC_Probabilities.csv"))
y_true = df_roc['True_Label'].values

# ==========================================
# 1. GENERATE 3 CONFUSION MATRICES (+ RAW CSV EXPORTS)
# ==========================================
for prefix, title in zip(models_list, model_titles):
    probs = df_roc[[f'{prefix}_Prob_{c}' for c in class_names]].values
    y_pred = np.argmax(probs, axis=1)

    cm = confusion_matrix(y_true, y_pred)

    # Export the raw confusion matrix to CSV for Origin/Prism/MATLAB plotting
    np.savetxt(os.path.join(output_dir, f"Raw_CM_{prefix}.csv"), cm, delimiter=",", fmt='%d')

    cm_norm = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis]

    fig, ax = plt.subplots(figsize=(5, 5))
    sns.heatmap(cm_norm, annot=cm, fmt='g', cmap='Blues', cbar=False,
                xticklabels=class_names, yticklabels=class_names,
                annot_kws={"size": 12, "weight": "bold"}, ax=ax)
    ax.set_xlabel('Predicted Phenotype', fontweight='bold')
    ax.set_ylabel('True Phenotype', fontweight='bold')
    ax.set_title(title, fontweight='bold', pad=15)
    plt.tight_layout()
    plt.savefig(os.path.join(output_dir, f"Fig_CM_{prefix}.pdf"), dpi=300)
    plt.close()

# ==========================================
# 2. GENERATE 4 ROC CURVES (ONE PER CLASS)
# ==========================================
for i, class_name in enumerate(class_names):
    y_true_bin = (y_true == i).astype(int)

    fig, ax = plt.subplots(figsize=(5, 5))
    for prefix, label, color in zip(models_list, model_titles, colors):
        y_scores = df_roc[f'{prefix}_Prob_{class_name}']
        fpr, tpr, _ = roc_curve(y_true_bin, y_scores)
        roc_auc = auc(fpr, tpr)
        ax.plot(fpr, tpr, color=color, lw=2, label=f'{label} (AUC = {roc_auc:.3f})')

    ax.plot([0, 1], [0, 1], 'k--', lw=1.5)
    ax.set_xlim([0.0, 1.0])
    ax.set_ylim([0.0, 1.05])
    ax.set_xlabel('False Positive Rate', fontweight='bold')
    ax.set_ylabel('True Positive Rate', fontweight='bold')
    ax.set_title(f'{class_name} Classification Accuracy', fontweight='bold', pad=15)
    ax.legend(loc="lower right", frameon=False, fontsize=9)
    sns.despine()
    plt.tight_layout()
    plt.savefig(os.path.join(output_dir, f"Fig_ROC_{class_name}.pdf"), dpi=300)
    plt.close()

# ==========================================
# 3. GENERATE 3 t-SNE PLOTS
# ==========================================
palette = ['#1f77b4', '#2ca02c', '#ff7f0e', '#d62728']

for prefix, title in zip(models_list, model_titles):
    df_tsne = pd.read_csv(os.path.join(data_dir, f"{prefix}_tSNE_Coordinates.csv"))

    fig, ax = plt.subplots(figsize=(5.5, 5))
    sns.scatterplot(x='t-SNE_1', y='t-SNE_2', hue='True_Class', palette=palette,
                    data=df_tsne, s=60, alpha=0.8, edgecolor='w', linewidth=0.5, ax=ax)
    ax.set_xlabel('t-SNE Dimension 1', fontweight='bold')
    ax.set_ylabel('t-SNE Dimension 2', fontweight='bold')
    ax.set_title(f'Feature Space: {title}', fontweight='bold', pad=15)
    ax.legend(title='Tissue Class', bbox_to_anchor=(1.05, 1), loc='upper left', frameon=False)
    sns.despine()
    plt.tight_layout()
    plt.savefig(os.path.join(output_dir, f"Fig_tSNE_{prefix}.pdf"), dpi=300, bbox_inches='tight')
    plt.close()

print(f"Success! All 10 vector figures (.pdf) and raw CM arrays (.csv) have been successfully generated.")